In [ ]:
import os
import numpy as np
import pandas as pd
import pickle

In [29]:
FIGURES_PATH = "figures"
os.makedirs(FIGURES_PATH,exist_ok=True)
# Set datafiles folder.
datafolder = "datafiles"
# Create directory in case it does not exist.
os.makedirs(datafolder,exist_ok=True)

# Set the name of the datafile.
datafile = "Dataset.npy"
# Load the dataset.
dataset = np.load(datafile)
# Define the spliter lambda function in order to tokenize the initial string 
# data.
spliter = lambda s: s.split(",")
# Apply the spliter function at each element of the dataset string array.
dataset = np.array([spliter(x) for x in dataset])

# Set the pickle file for storing the initial dataframe.
pickle_file = os.path.join(datafolder,"dataframe.pkl")
# Check the existence of the specifiied file.
#read pickled datafile
if os.path.exists(pickle_file):
    # Load the pickle file.
    dataframe = pd.read_pickle(pickle_file)
else:
    # Create the dataframe object.
    dataframe = pd.DataFrame(dataset, columns=["user","item","rating","date"])
    # Convert the string elements of the "users" series into integers.
    dataframe["user"] = dataframe["user"].apply(lambda s:np.int64(s.replace("ur","")))
    # Convert the string elements of the "items" series into integers.
    dataframe["item"] = dataframe["item"].apply(lambda s:np.int64(s.replace("tt","")))
    # Conver the string elements of the "ratings" series into integers.
    dataframe["rating"] = dataframe["rating"].apply(lambda s:np.int64(s))
    # Convert the string elements of the "dates" series into datetime objects.
    dataframe["date"] = pd.to_datetime(dataframe["date"])
    dataframe.to_pickle(pickle_file)

In [30]:
#find and remove duplicates and remove date column
duplicates=dataframe.duplicated().sum()
if (duplicates>0):
    print("Number of duplicate values:",duplicates)
    dataframe.drop_duplicates(inplace=True)
dataframe.drop(columns=["date"],inplace=True)

dataframe.to_pickle(pickle_file)

dataframe

Number of duplicate values: 9382


,user,item,rating
0,4592644,120884,10
1,3174947,118688,3
2,3780035,387887,8
3,4592628,346491,1
4,3174947,94721,8
...,...,...,...
4669815,581842,107977,6
4669816,3174947,103776,8
4669817,4592639,107423,9
4669818,4581944,102614,8


In [31]:

def get_stats(dataframe):
    # Get the unique users in the dataset.
    users = dataframe["user"].unique()
    # Get the number of unique users.
    users_num = len(users)
    # Get the unique items in the dataset.
    items = dataframe["item"].unique()
    items_num = len(items)
    # Get the total number of existing ratings.
    ratings_num = dataframe.shape[0]
    # Report the number of unique users and items in the dataset.
    print("DATASET: {0} number of unique users and {1} of unique items".format(users_num,items_num))
    # Report the total number of existing ratings in the dataset.
    print("DATASET: {} total number of existing ratings".format(ratings_num))
    return users_num,items_num,ratings_num
users_num,items_num,ratings_num = get_stats(dataframe)

DATASET: 1499238 number of unique users and 351109 of unique items
DATASET: 4660438 total number of existing ratings


In [44]:
#Now we  need to choose a subset of the ratings in the dataset that corresponded to users and items within a specific range of ratings.
#We also need to ensure that by the end of the process there are not any users that have rated too few items or items that have been rated too few times.

In [32]:
minimum_user_ratings = 100
maximum_user_ratings = 300
minimum_item_ratings = 10

new_df = dataframe.copy()
# Initial filtering
def filter_users_and_items(df):
    # Filter users
    user_ratings_count = df.groupby("user")["rating"].count().sort_values(ascending=
                                    False).reset_index(name="ratings_num")
    filtered_users = user_ratings_count.loc[(user_ratings_count["ratings_num"] >= minimum_user_ratings) & 
                                            (user_ratings_count["ratings_num"] <= maximum_user_ratings)]
    df = df.loc[df["user"].isin(filtered_users["user"])]
    print("Filtered users: {} -> {}".format(user_ratings_count.shape[0], filtered_users.shape[0]))

    # Filter items
    item_ratings_count = df.groupby("item")["rating"].count().sort_values(ascending=
                                    False).reset_index(name="users_rated")
    filtered_items = item_ratings_count.loc[item_ratings_count["users_rated"] >= minimum_item_ratings]
    df = df.loc[df["item"].isin(filtered_items["item"])]
    print("Filtered items: {} -> {}".format(item_ratings_count.shape[0], filtered_items.shape[0]))

    return df

# Iteratively filter users and items until no more changes
previous_shape = None
current_shape = new_df.shape

while previous_shape != current_shape:
    previous_shape = current_shape
    new_df = filter_users_and_items(   new_df)
    current_shape = new_df.shape

# Reset index and drop the old index column
final_df = new_df.reset_index(drop=True)

# Get final stats
users_num, items_num, ratings_num = get_stats(final_df)



Filtered users: 1499238 -> 2290
Filtered items: 91878 -> 7078
Filtered users: 2249 -> 891
Filtered items: 7064 -> 4055
Filtered users: 891 -> 640
Filtered items: 4055 -> 3214
Filtered users: 640 -> 535
Filtered items: 3214 -> 2792
Filtered users: 535 -> 473
Filtered items: 2792 -> 2555
Filtered users: 473 -> 444
Filtered items: 2555 -> 2439
Filtered users: 444 -> 420
Filtered items: 2439 -> 2330
Filtered users: 420 -> 410
Filtered items: 2330 -> 2282
Filtered users: 410 -> 397
Filtered items: 2282 -> 2226
Filtered users: 397 -> 388
Filtered items: 2226 -> 2177
Filtered users: 388 -> 380
Filtered items: 2177 -> 2142
Filtered users: 380 -> 370
Filtered items: 2142 -> 2089
Filtered users: 370 -> 363
Filtered items: 2089 -> 2046
Filtered users: 363 -> 355
Filtered items: 2046 -> 2010
Filtered users: 355 -> 347
Filtered items: 2010 -> 1975
Filtered users: 347 -> 343
Filtered items: 1975 -> 1963
Filtered users: 343 -> 341
Filtered items: 1963 -> 1957
Filtered users: 341 -> 341
Filtered items

In [33]:
#find and remove duplicates and remove date column
duplicates=final_df.duplicated().sum()
if (duplicates>0):
    print("Number of duplicate values:",duplicates)
    final_df.drop_duplicates(inplace=True)

Number of duplicate values: 79


In [34]:
# Get the unique users and items in the final dataframe along with the final
# number of ratings.
final_users = final_df["user"].unique()
final_items = final_df["item"].unique()
final_users_num = len(final_users)
final_items_num = len(final_items)
final_ratings_num = len(final_df)

# Report the final number of unique users and items in the dataset.
print("REDUCED DATASET: {0} number of unique users and {1} of unique items".format(final_users_num,final_items_num))
# Report the final  number of existing ratings in the dataset.
print("REDUCED DATASET: {} total number of existing ratings".format(final_ratings_num))

REDUCED DATASET: 341 number of unique users and 1957 of unique items
REDUCED DATASET: 46196 total number of existing ratings


In [35]:
# We need to reset the users and items ids in order to be able to construct the
# networks of users and items. Users and Items ids should be consecutive integers
# in the [1...final_users_num] and [1...final_items_num].
# Initialy, we need to acquire the sorted versions of the user and item ids.
sorted_final_users = np.sort(final_users)
sorted_final_items = np.sort(final_items)


# Generate the dictionary of final users as a mapping of the following form:
# sorted_final_users --> [0...final_users_num-1]
final_users_dict = dict(zip(sorted_final_users,list(range(0,final_users_num))))
# Generate the dictionary of final items as a mapping of the following form:
# sorted_final_items --> [0...final_items_num-1]
final_items_dict = dict(zip(sorted_final_items,list(range(0,final_items_num))))
# Apply the previously defined dictionary-based maps on the users and item 
# columns of the final dataframe.
final_df["user"] = final_df["user"].map(final_users_dict)
final_df["item"] = final_df["item"].map(final_items_dict)

final_df

,user,item,rating
0,0,312,7
1,31,751,9
2,5,597,6
3,5,590,6
4,0,513,6
...,...,...,...
46270,69,525,5
46271,56,700,10
46272,1,671,6
46273,0,464,6


In [36]:
#set the path for the final dataframe
final_df_path = os.path.join(datafolder,"final_df.pkl")
#check if the file exists
if os.path.exists(final_df_path):
    #load the final dataframe
    final_df = pd.read_pickle(final_df_path)
else:
    #save the final dataframe
    final_df.to_pickle(final_df_path)